In [1]:
%matplotlib inline
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import numpy as np
import pandas as pd

_PATH = Path.cwd()
_PROJECT_ROOT = _PATH
for _ in range(5):
    if (_PROJECT_ROOT / ".env").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent

sys.path.insert(0, str(_PATH))
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(_PROJECT_ROOT / ".env")

def get_engine():
    host = os.getenv("PGHOST", "localhost")
    port = os.getenv("PGPORT", "5432")
    user = os.getenv("PGUSER", "postgres")
    password = os.getenv("PGPASSWORD", "")
    dbname = os.getenv("PGDATABASE", "baseball")
    pw = quote_plus(password) if password else ""
    return create_engine(f"postgresql://{user}:{pw}@{host}:{port}/{dbname}")

from data_prep_batters import OUTCOME_MAPPING, BUCKET_CLASS_NAMES, NUM_BUCKET_CLASSES
engine = get_engine()
print("Setup OK. OUTCOME_MAPPING and BUCKET_CLASS_NAMES loaded.")

Setup OK. OUTCOME_MAPPING and BUCKET_CLASS_NAMES loaded.


In [2]:
# Load in-play rows: launch_speed and launch_angle required (batted balls only)
q = """
SELECT release_speed, plate_x, plate_z, launch_speed, launch_angle, events,
       career_SLG AS career_slg, game_date, game_year
FROM clean_statcast_with_batter
WHERE launch_speed IS NOT NULL AND launch_angle IS NOT NULL
"""
df = pd.read_sql(q, engine)
df["career_slg"] = pd.to_numeric(df["career_slg"], errors="coerce")

# Map events to outcome bucket (0-4); drop unmapped
df["events_clean"] = df["events"].astype(str).str.strip().str.lower()
df["outcome_bucket"] = df["events_clean"].map(OUTCOME_MAPPING)
df = df.dropna(subset=["outcome_bucket"]).copy()
df["outcome_bucket"] = df["outcome_bucket"].astype(int)

# Fill missing career_slg with median
if (df["career_slg"] < 0).all() or df["career_slg"].isna().all():
    df["batter_power"] = 0.4
else:
    med = df.loc[df["career_slg"] >= 0, "career_slg"].median()
    df["batter_power"] = df["career_slg"].replace(-1, np.nan).fillna(med)

# Features X: pitch_mph, plate_x, plate_z, one-hot outcome (5), batter_power
X = pd.DataFrame({
    "pitch_mph": df["release_speed"].astype(float),
    "plate_x": df["plate_x"].astype(float).fillna(0),
    "plate_z": df["plate_z"].astype(float).fillna(0),
    "batter_power": df["batter_power"].astype(float),
})
for k in range(NUM_BUCKET_CLASSES):
    X[f"outcome_{k}"] = (df["outcome_bucket"] == k).astype(int)

# Targets y: launch_speed, launch_angle (no hc_x/hc_y so no spray_angle)
y = df[["launch_speed", "launch_angle"]].astype(float)
X.index = df.index
y.index = df.index
feature_cols = list(X.columns)
print(f"In-play rows: {len(X)}. Features: {feature_cols[:4]} + one-hot outcome (5). Targets: launch_speed, launch_angle")
print("Outcome bucket counts:\n", df["outcome_bucket"].value_counts().sort_index())

In-play rows: 362034. Features: ['pitch_mph', 'plate_x', 'plate_z', 'batter_power'] + one-hot outcome (5). Targets: launch_speed, launch_angle
Outcome bucket counts:
 outcome_bucket
0    241551
3     77779
4     42704
Name: count, dtype: int64


In [3]:
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
from temporal_split import temporal_train_val_test

train_df, val_df, test_df, split_meta = temporal_train_val_test(df, train_frac=0.7, val_frac=0.15)
print("Temporal split:", split_meta)
X_train = X.loc[train_df.index]
y_train = y.loc[train_df.index]
X_val = X.loc[val_df.index]
y_val = y.loc[val_df.index]
X_test = X.loc[test_df.index]
y_test = y.loc[test_df.index]

# Multi-output regressor: predict launch_speed, launch_angle
reg = MultiOutputRegressor(xgb.XGBRegressor(n_estimators=100, max_depth=5, random_state=42))
reg.fit(X_train[feature_cols], y_train)
y_pred_val = reg.predict(X_val[feature_cols])
y_pred_test = reg.predict(X_test[feature_cols])

for i, name in enumerate(["launch_speed", "launch_angle"]):
    mae = mean_absolute_error(y_val.iloc[:, i], y_pred_val[:, i])
    rmse = np.sqrt(mean_squared_error(y_val.iloc[:, i], y_pred_val[:, i]))
    print(f"Val {name}: MAE = {mae:.2f}, RMSE = {rmse:.2f}")
print("Regressor fitted.")

Temporal split: {'years_present': [2023, 2024, 2025], 'split_kind': 'multi_season', 'val_year': 2024, 'test_year': 2025, 'n_train': 120406, 'n_val': 120383, 'n_test': 121245}
Val launch_speed: MAE = 9.76, RMSE = 12.97
Val launch_angle: MAE = 20.89, RMSE = 27.51
Regressor fitted.


In [4]:
# Save regressor and metadata for Phase 3 (simulator)
import json
import joblib

_SAVED = Path(_PATH) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
joblib.dump(reg, _SAVED / "outcome_regressor.joblib")
metadata = {
    "feature_cols": feature_cols,
    "target_names": ["launch_speed", "launch_angle"],
    "n_outcomes": NUM_BUCKET_CLASSES,
    "outcome_names": BUCKET_CLASS_NAMES,
}
with open(_SAVED / "outcome_regressor_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("Saved outcome_regressor.joblib and outcome_regressor_metadata.json")

Saved outcome_regressor.joblib and outcome_regressor_metadata.json


In [5]:
# BIP-only launch regressor (no outcome one-hot): for 3-bucket simulator when classifier says "in_play"
# Features: pitch_mph, plate_x, plate_z, batter_power only. Targets: launch_speed, launch_angle.
bip_X = X[["pitch_mph", "plate_x", "plate_z", "batter_power"]].copy()
bip_feature_cols = list(bip_X.columns)
bip_y = y.copy()

bip_train_idx = train_df.index.intersection(bip_X.index)
bip_val_idx = val_df.index.intersection(bip_X.index)
bip_test_idx = test_df.index.intersection(bip_X.index)
bip_X_train, bip_y_train = bip_X.loc[bip_train_idx], bip_y.loc[bip_train_idx]
bip_X_val, bip_y_val = bip_X.loc[bip_val_idx], bip_y.loc[bip_val_idx]
bip_X_test, bip_y_test = bip_X.loc[bip_test_idx], bip_y.loc[bip_test_idx]

reg_bip = MultiOutputRegressor(xgb.XGBRegressor(n_estimators=100, max_depth=5, random_state=42))
reg_bip.fit(bip_X_train[bip_feature_cols], bip_y_train)
bip_pred_val = reg_bip.predict(bip_X_val[bip_feature_cols])
bip_pred_test = reg_bip.predict(bip_X_test[bip_feature_cols])
for i, name in enumerate(["launch_speed", "launch_angle"]):
    mae = mean_absolute_error(bip_y_val.iloc[:, i], bip_pred_val[:, i])
    print(f"BIP Val {name}: MAE = {mae:.2f}")

# Ridge baseline (same BIP features) for plan Item 3 comparison
ridge_bip = MultiOutputRegressor(Ridge(alpha=1.0, random_state=42))
ridge_bip.fit(bip_X_train[bip_feature_cols], bip_y_train)
ridge_bip_pred_test = ridge_bip.predict(bip_X_test[bip_feature_cols])
rmse_bip_speed = np.sqrt(mean_squared_error(bip_y_test.iloc[:, 0], bip_pred_test[:, 0]))
rmse_ridge_speed = np.sqrt(mean_squared_error(bip_y_test.iloc[:, 0], ridge_bip_pred_test[:, 0]))
rmse_gain = 100.0 * ((rmse_ridge_speed - rmse_bip_speed) / rmse_ridge_speed) if rmse_ridge_speed > 0 else 0.0
print(f"BIP Test launch_speed RMSE (XGB): {rmse_bip_speed:.2f}")
print(f"BIP Test launch_speed RMSE (Ridge baseline): {rmse_ridge_speed:.2f}")
print(f"BIP launch_speed RMSE improvement vs Ridge: {rmse_gain:.1f}%")
print("BIP regressor fitted.")

BIP Val launch_speed: MAE = 10.44
BIP Val launch_angle: MAE = 21.76
BIP regressor fitted.


In [6]:
# Save BIP-only regressor and metadata
_SAVED = Path(_PATH) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)
_MODELS = _PROJECT_ROOT / "Models" / "saved_models"
_MODELS.mkdir(parents=True, exist_ok=True)
joblib.dump(reg_bip, _SAVED / "bip_launch_regressor.joblib")
bip_metadata = {
    "feature_cols": bip_feature_cols,
    "target_names": ["launch_speed", "launch_angle"],
}
with open(_SAVED / "bip_launch_regressor_metadata.json", "w") as f:
    json.dump(bip_metadata, f, indent=2)
print("Saved bip_launch_regressor.joblib and bip_launch_regressor_metadata.json")
if _MODELS.exists():
    import shutil
    shutil.copy(_SAVED / "bip_launch_regressor.joblib", _MODELS / "bip_launch_regressor.joblib")
    shutil.copy(_SAVED / "bip_launch_regressor_metadata.json", _MODELS / "bip_launch_regressor_metadata.json")
    print("Copied to Models/saved_models")

Saved bip_launch_regressor.joblib and bip_launch_regressor_metadata.json
Copied to Models/saved_models
